#### Part Two: Advanced metahuristics
"Parts are going to be in seperate notbooks for clarity also helper functions wont be stored in a single file to import for going back and forth from notebook would be annoying so i will reinitialze them here too btw classes used to make code modular not used because it makes one section of notebook long i could register them in one place"

In [9]:
import pandas as pd
import numpy as np
import time 
import matplotlib.pyplot as plt
import mplcursors
import seaborn as sns
import mplcursors
import ipympl
import sys
sys.path.append("../helper")
from workload_data_loader import load_workload_instance as ld
from itertools import combinations
from scipy.stats import friedmanchisquare
print(ipympl.__version__)

0.10.0


Helper functions for one instance
- fitness(x):evaluation of total utility
- is_feasible(x):evaluation of constraints passed over or not

In [10]:
def fitness(x:np.ndarray,p:np.ndarray):
   return p @ x

def is_feasible(x:np.ndarray,r:np.ndarray,b:np.ndarray):
    return np.all(r @ x <= b)

The approach taken will be a table of suggested approaches from literature regarding all the components of the genetic algorithm each table will consist of:
- The approach
- Reasoning`
- Supported literature

Although i will test them on my own and compare them too rather than blindly focusing on literature the first approach to be explored will be the Chu & Beasley (1998), A genetic algorithm for the multidimensional knapsack problem, Journal of Heuristics. as it is the most similar for the foundation then the above criteria will ensue

Initialization of population:
| Approach  | Reasoning | Supported Literature      |Infeasible Allowed |
|-------|-----|-----------|----
| Random Initialization | Simple Diverse Solutions(May become less diverse on infeasibiliity)  | ChChu & Beasley (1998)     | No|
| Bob   | 30  | London    ||
| Carol | 28  | New York  ||

In [ ]:
# The Chu and beasely uses a randomization algorithm to generate a feasible solution, it is not guaranteed to generate a feasible solution but it is very likely to do ss.
# It goes through step by step adding of random items instead of all at once and checks feasabilty at each step
# It stops at first non found variable

# We dont need size because in these problems demands is same shape
def initialize_random_chu(demands:np.ndarray,capacity,rng):    
    # this is the random generation
    size = demands.shape[1]
    # initialize 0 items chosen
    state = np.zeros(size,dtype=int)
    # This gives a random permuation from the size the values will be the item number we select [] 
    items = rng.permutation(size)
    for i in items:
        candidate = state.copy()
        candidate[i] = 1
        if is_feasible(candidate,demands,capacity):
            state[i] = 1
        else:
            break
    return state


# This will actually create the whole population 
# Put is feasible as a param later on
def initialize_population(demands,capacity,initializer_func,seed=42,isFeasible=True,size=30):
    # reproducible
    rng = np.random.default_rng(seed)
    # No dublicates for now in the population, this is not a problem because the population size is small and the solution space is large
    population = set()
    
    while len(population) < size:
        individual = initializer_func(demands,capacity,rng)
        population.add(tuple(individual))
    
    return np.array(list(population))
    




Evaluation: 
| Approach  | Reasoning | Supported Literature      |
|-------|-----|-----------|
| Using Fitness| Simple alligns with no non feasible values  | ChChu & Beasley (1998)     

In [ ]:
# Ruse the fitness helper at top for evaluation

Selection of parents:
| Approach  | Reasoning | Supported Literature      |
|-------|-----|-----------|
| Using Fitness,Tournament Selection size-2 parents-2| Simple alligns with no non feasible values  | ChChu & Beasley (1998)    

In [ ]:

def selection_tournament(population,fitness,rng,tournament_size=2,parents_size=2):
    # We will store the indices of the parents selected
    parents = set()
    
    while len(parents) < parents_size:
        # We will use the rng to select two individuals randomly
        # Replace false gives us distinct values
        contestants = rng.choice(len(population),size=tournament_size,replace=False)
        
        # Slection of the individual with highest fitness so the array of the highest individual will be chosen
        winner = contestants[np.argmax(fitness[contestants])]
        parents.append(winner)
    
    # Returns the index of the parents
    return np.array(list(parents))

Crossover:
| Approach  | Reasoning | Supported Literature      |
|-------|-----|-----------|
| Unifrom (mask is random using our seed)| More flexible, but can produce very different offsprigns | ChChu & Beasley (1998)    

In [ ]:
def crossover_uniform_randommask(parent1,parent2,rng):
    # Create the random mask
    mask = np.random.random_integers(0,2,size=len(parent1))
    
    # Use a filter to get offspring
    return np.where(mask == 1,parent1,parent2)


Muatation Function:
| Approach  | Reasoning | Supported Literature      |
|-------|-----|-----------|
| Simple sigle bit flip| Simple  | ChChu & Beasley (1998)    

In [ ]:
# ALthough common approach is for mutation and the mutation
def mutation_single_bit_flip_chu(child,rng,proability=0.2):
    # Here the probability is tied to the bits inside the mutation function not outside 
    # Copied cause dont want to change outside
    mutated = child.copy()

    # Select bits like this first i gives random value 0-1 if these are less than the probability they get a True boolean
    # we can then use this mask to change those bits
    i = rng.integers(len(mutated)) < proability

    # Flip the bit
    mutated[i] = 1 - mutated[i]

    return mutated




# This is going to be the abstraction that adds the proabability and muation on top here
def apply_mutation(child,rng,muatation_func,mutation_prob):
    if rng.random < mutation_prob:
        child = muatation_func(child,rng)
    return child

In [12]:
n, m, utilities, demands, capacities = ld(r"..\data\multi_resource_n90_m5.txt")
print("=== Dataset Ingestion Summary ===")
print(f"Number of candidate models (n) : {n}")
print(f"Number of resource bounds (m)  : {m}")
print(f"Utility scores shape (p_j)     : {utilities.shape}")
print(f"Resource demands shape (r_ij)  : {demands.shape}")
print(f"Capacity limits (b_i)          : {capacities}")


=== Dataset Ingestion Summary ===
Number of candidate models (n) : 90
Number of resource bounds (m)  : 5
Utility scores shape (p_j)     : (90,)
Resource demands shape (r_ij)  : (5, 90)
Capacity limits (b_i)          : [2100. 1100. 3300. 3700. 3600.]


In [27]:
arr = initialize_population(demands,capacities,initialize_random_chu) @ demands.T
print(arr.shape)
df = pd.DataFrame(arr)
display(df)

(30, 5)


,0,1,2,3,4
0,1227.0,1095.0,1245.0,1110.0,1259.0
1,1162.0,1045.0,1029.0,1180.0,1118.0
2,1168.0,1079.0,1392.0,1485.0,1394.0
3,1112.0,1087.0,1398.0,1219.0,1253.0
4,801.0,1085.0,908.0,1040.0,1343.0
5,1303.0,1068.0,1292.0,1334.0,1295.0
6,815.0,1070.0,1206.0,1110.0,1279.0
7,1012.0,1072.0,1315.0,1541.0,1043.0
8,1203.0,1094.0,1343.0,1471.0,1323.0
9,1016.0,1063.0,1180.0,1241.0,982.0
